In [1]:
import pandas as pd
import baseline_simulator
from utils import *
from pathlib import Path

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

In [ ]:
sessions_file = Path(__name__).resolve().parents[1] / "data" / "Sessions3.csv"
sessions_df = pd.read_csv(sessions_file)

sessions_df = sessions_df.sort_values(by="startChargeTime")

for month in range(1, 2):
    test_df = sessions_df[
        (pd.to_datetime(sessions_df["connectTime"]).dt.year == 2023)
        & (pd.to_datetime(sessions_df["connectTime"]).dt.month == month)
    ]

    test_df = test_df[test_df["DurationHrs"] > 0.5]
    test_df = test_df[test_df["cumEnergy_Wh"] > 0]
    test_df["choice"] = "SCHEDULED"

    sim = baseline_simulator.BaselineSimulator(
        test_df,
        verbose=False,
    )

    power_profiles, prices, hourly_prices = sim.simulate()
    session_results = get_session_results(
        test_df, power_profiles, prices, sim.TOU, sim.delta_t
    )
    session_results.to_csv(f"results/{month}-2023-all-scheduled.csv")

    agg_power_profile = aggregate_power_profiles(test_df, power_profiles, sim.delta_t)
    charging_revenue, TOU_costs = get_profit(
        test_df, power_profiles, prices, sim.delta_t, sim.TOU
    )

    demand_charge_kwh = max(agg_power_profile)
    demand_charge_cents = sim.cost_dc * demand_charge_kwh

    print("------------------------------------------------------------")

    print("Month", month)
    print("Total Profit", charging_revenue - TOU_costs - demand_charge_cents)
    print("Charging Revenue", charging_revenue)
    print("TOU Costs", TOU_costs)
    print("Demand Charge Costs (cents)", demand_charge_cents)
    print("Peak Power", demand_charge_kwh)

In [ ]:
sessions_df

In [ ]:
agg_power_profile_all_sch = aggregate_power_profiles(
    test_df, power_profiles, sim.delta_t
)

profit_all_sch = get_profit(test_df, power_profiles, prices, sim.delta_t, sim.TOU)

profit_all_sch - sim.cost_dc * max(agg_power_profile_all_sch), max(
    agg_power_profile_all_sch
)